# **Datalab II Sprint 3 2025 - 2026**
**Plaats:** De Haagse Hogenschool, ADS & AI<br>
**Auteurs:** M. Kilinc, D. Hoogenbosch, J. Wolthuis, S. Slingerland, L. van Hamersveld<br>
**Groep:** B2 <br>
**Coach:** Onur Tezel <br>
**Datum:** 31/03/2026


| Naam  | Studentnummer |
|-------|---------------|
| Lucas | 25076116      |
| Sandro| 25154370      |
| Memhet| 25135007      |
| Julius| 25090216      |
| Dylan | 25118498      |
---

## **Inhoudsopgave**
1. [Imports & Configuratie](#1)
2. [Data inladen](#2)
3. [Data Exploration met SQL](#3)
4. [Onderzoeken Teameigenschappen](#4)

---
<a id='1'></a>
## **1. Imports & Configuratie**

In [5]:
# Imports
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Nieuwe visualisatie bibliotheken
import plotly.express as px
import nbformat

In [6]:
# Via deze functie in pandas zorgen we ervoor dat we de DataFrames volledig kunnen weergeven in de cell
pd.set_option('display.max_rows', 200)

---
<a id='2'></a>
## **2. Data inladen**
In de onderstaande kolommen laden we de data op dezelfde manier als de vorige keer, namelijk via de Kaggle API. <br> 
Hieronder vind je een uitleg over het gebruik en een referentie.



### **2.1 Gebruik van de Kaggle API** 
We gebruiken de Kaggle API om de data lokaal op te slaan en dit hoeft slechts eenmalig te gebeuren.<br>Download de API-sleutel uit onze GitHub-repository en plaats deze in de map C:\Users\<Jouw_gebruikersnaam>\.kaggle. <br>

Raadpleeg voor verdere instructies de officiële documentatie op https://www.kaggle.com/docs/api.<br>
Verwijder eventueel het # in het script, voer de pip install en de data-import uit en je bent klaar om de data lokaal te gebruiken.


In [7]:
# %pip install kaggle 
import kaggle 
kaggle.api.authenticate()
kaggle.api.dataset_download_files("hugomathien/soccer", path='.', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer


In [8]:
# Maak een verbinding met de SQLite-database "database.sqlite"
connection = sqlite3.connect("database.sqlite")

# Voer een SQL-query uit om alle gegevens uit de tabel 'Match' op te halen, het resultaat wordt ingelezen als pandas DataFrame
df = pd.read_sql("""
    
    SELECT *  
        FROM Match 
                 
    """, 
    connection)

---
<a id='3'></a>
## **3. Data Exploration met SQL**

### 3.1 Aantal gespeelde wedstrijden per seizoen

In [9]:
AZ_API_ID = 10229
AZ_FIFA_ID = 1906

# Query
query  = f"""

SELECT season, COUNT(*) AS aantal_gespeelde_wedstrijden
FROM Match

WHERE home_team_api_id = {AZ_API_ID}
   OR away_team_api_id = {AZ_API_ID}

GROUP BY season
ORDER BY season;

"""
# Opslaan data uit query in DataFrame
df = pd.read_sql(query, connection)

# DataFrame
display(df)

# Printen samenvatting ondervindingen 
print(f'AZ alkmaar is sinds seizoen 1998/1999 onafgebroken in de Eredivisie een speelde: {df["aantal_gespeelde_wedstrijden"].sum()} wedstrijden')


,season,aantal_gespeelde_wedstrijden
0,2008/2009,34
1,2009/2010,34
2,2010/2011,34
3,2011/2012,34
4,2012/2013,34
5,2013/2014,34
6,2014/2015,34
7,2015/2016,34


AZ alkmaar is sinds seizoen 1998/1999 onafgebroken in de Eredivisie een speelde: 272 wedstrijden


---
### 3.2 Aantal gespeelde wedstrijden kalender jaar 2010

In [10]:
# In deze query wordt gebruik gemaakt van strftime (string format time) 
# Deze functie converteert datum en tijdobjecten naar leesbare tekenreeksen op basis van specifieke opmaakcodes

# Query
query  = f"""

SELECT season, COUNT(*) AS gespeelde_wedstrijden
FROM Match

WHERE (home_team_api_id = {AZ_API_ID} OR 
away_team_api_id = {AZ_API_ID})
AND strftime('%Y', date) = '2010' 

GROUP BY season
ORDER BY season;

"""

# Opslaan data uit query in DataFrame
df = pd.read_sql(query, connection)

# DataFrame
display(df)

# Printen samenvatting ondervindingen 
print(f'In het kalender jaar 2010 speelde AZ: {df["gespeelde_wedstrijden"].sum()} wedstrijden')

,season,gespeelde_wedstrijden
0,2009/2010,16
1,2010/2011,18


In het kalender jaar 2010 speelde AZ: 34 wedstrijden


---
### 3.3 Punten per seizoen & eindklassering

In [11]:
# Ik gebruik hier UNION ALL deze functie stapelt rijen van tabellen op elkaar inclusief dubbele records
Eredivisie_id = 13274

query = f"""
SELECT
    T.team_long_name AS Team,
    S.season,
    SUM(S.punten) AS totaal_punten
FROM (
    SELECT
        season,
        home_team_api_id AS team_api_id,
        CASE
            WHEN home_team_goal > away_team_goal THEN 3
            WHEN home_team_goal = away_team_goal THEN 1
            ELSE 0
        END AS punten
    FROM Match
    WHERE league_id = {Eredivisie_id}

    UNION ALL

    SELECT
        season,
        away_team_api_id AS team_api_id,
        CASE
            WHEN away_team_goal > home_team_goal THEN 3
            WHEN away_team_goal = home_team_goal THEN 1
            ELSE 0
        END AS punten

    FROM Match
    WHERE league_id = {Eredivisie_id}
) AS S

JOIN Team AS T ON S.team_api_id = T.team_api_id
GROUP BY S.season, S.team_api_id, T.team_long_name
ORDER BY S.season DESC, totaal_punten DESC;
"""

# Opslaan data uit query in DataFrame
df_2 = pd.read_sql(query, connection)

# DataFrame
display(df_2)

,Team,season,totaal_punten
0,PSV,2015/2016,84
1,Ajax,2015/2016,82
2,Feyenoord,2015/2016,63
3,AZ,2015/2016,59
4,FC Utrecht,2015/2016,53
5,Heracles Almelo,2015/2016,51
6,FC Groningen,2015/2016,50
7,PEC Zwolle,2015/2016,48
8,Vitesse,2015/2016,46
9,N.E.C.,2015/2016,46


---
### 3.4 Eindklassering AZ per seizoen

In [15]:
# Rangschikking per seizoen bepalen op basis van totaal punten
df_2['klassering'] = df_2.groupby('season')['totaal_punten'].rank(ascending=False, method='min').astype(int)

# Eindklassering van AZ Alkmaar per seizoen weergeven
df_az_klassering = df_2[df_2['Team'] == 'AZ'][['season', 'totaal_punten', 'klassering']].sort_values('season').reset_index(drop=True)
display(df_az_klassering)

print(f'AZ Alkmaar eindigde gemiddeld op plaats {df_az_klassering["klassering"].mean():.1f} in de Eredivisie')

,season,totaal_punten,klassering
0,2008/2009,80,1
1,2009/2010,62,5
2,2010/2011,59,4
3,2011/2012,65,4
4,2012/2013,39,10
5,2013/2014,47,8
6,2014/2015,62,3
7,2015/2016,59,4


AZ Alkmaar eindigde gemiddeld op plaats 4.9 in de Eredivisie


---
<a id='4'></a>
## **4. Onderzoeken Teameigenschappen**

---
### **4.1 Teameigenschappen samenvoegen**

In [16]:
query = """
SELECT
    T.team_long_name AS Team,
    AVG(TA.buildUpPlaySpeed)            AS buildUpPlaySpeed,
    AVG(TA.buildUpPlayDribbling)        AS buildUpPlayDribbling,
    AVG(TA.buildUpPlayPassing)          AS buildUpPlayPassing,
    AVG(TA.chanceCreationPassing)       AS chanceCreationPassing,
    AVG(TA.chanceCreationCrossing)      AS chanceCreationCrossing,
    AVG(TA.chanceCreationShooting)      AS chanceCreationShooting,
    AVG(TA.defencePressure)             AS defencePressure,
    AVG(TA.defenceAggression)           AS defenceAggression,
    AVG(TA.defenceTeamWidth)            AS defenceTeamWidth
FROM Team_Attributes AS TA
JOIN Team AS T ON TA.team_api_id = T.team_api_id
GROUP BY T.team_long_name;
"""

# Opslaan data uit query in DataFrame
df_attrs = pd.read_sql(query, connection)

# Samenvoegen met het punten DataFrame op teamnaam
df_merged = pd.merge(df_2, df_attrs, on='Team', how='inner')

# DataFrame
display(df_merged.head(10))

print(f'Samengevoegd DataFrame: {len(df_merged)} rijen, {len(df_merged.columns)} kolommen')

,Team,season,totaal_punten,klassering,buildUpPlaySpeed,buildUpPlayDribbling,buildUpPlayPassing,chanceCreationPassing,chanceCreationCrossing,chanceCreationShooting,defencePressure,defenceAggression,defenceTeamWidth
0,PSV,2015/2016,84,1,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
1,PSV,2014/2015,88,1,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
2,PSV,2013/2014,59,4,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
3,PSV,2012/2013,69,2,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
4,PSV,2011/2012,69,3,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
5,PSV,2010/2011,69,3,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
6,PSV,2009/2010,78,3,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
7,PSV,2008/2009,65,4,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
8,Ajax,2015/2016,82,2,35.166667,41.5,33.833333,50.666667,58.5,49.666667,59.833333,53.833333,54.333333
9,Ajax,2014/2015,71,2,35.166667,41.5,33.833333,50.666667,58.5,49.666667,59.833333,53.833333,54.333333


Samengevoegd DataFrame: 143 rijen, 13 kolommen
